# 03 Authoring Libraries
A library written in a `%%cql` cell, a measure in a second cell that `include`s it, both run from Python,
edited in place, and published to a folder. Then the examples' own folder of measures, mounted the way
a Databricks Git Folder of your measures would be.

It assumes **01 Getting Started**: the magic, `cql()` and the result shape.

**Runs on serverless — nothing to attach, nothing to configure.** Just **Run All**. The one
prerequisite is a workspace where CQL on Databricks is deployed.

## Load the magic
**Expected output:** nothing. The cell is silent on success; an error here means the notebook is
not on the **CQL on Databricks** base environment.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

In [0]:
# Start clean, so Run All gives the same output however often it runs: drop any earlier mount and any
# library a `%%cql` cell declared (CLIENT-SESSION-RESET). Section 6 redefines a library in place, so a
# stale draft from an earlier run would otherwise answer the first include.
from cql_databricks.compiler import reset

reset(spark)

## 1. The data a measure reads
The measures below read FHIR data. Every `[Resource]` retrieve reads the default environment's table of its
type, in its `data` schema, so there is nothing to load. Authoring the logic in cells changes nothing about how
data reaches it — that is the seam holding still while everything above it moves.

**Expected output:** the environment's name and catalog, then one line per type: its row count and the table
it is read from.

In [0]:
from cql_databricks.resolvers import EnvironmentResolver
from cql_databricks.data_bindings import spark_runner

environment = EnvironmentResolver(spark_runner(spark)).default()
print("environment:", environment.name, "in catalog", environment.catalog)
for t in ("Condition", "Encounter", "Patient"):
    view = f"{environment.schema('data')}.{t.lower()}"
    print(f"{t:12} {spark.table(view).count()} rows -> {view}")

## 2. Connect — with nothing mounted as a library source
A client of your own, from `connect()`, with no `library_path`. There is no folder of CQL to point at,
because the libraries are about to be written in the next two cells.

**Expected output:** `libraries declared so far: []` — an empty session, which is the honest state
of a notebook before anything has been authored.

In [0]:
from cql_databricks.compiler import connect

client = connect(spark)
print("libraries declared so far:", [str(r) for r in client.library_provider.declared()])

## 3. A library in a cell — it declares, it does not run
A helper library first: what does it mean for a patient to have visits during the measurement period?
It goes in its own library because "had an encounter" is not specific to one measure — the next measure
you write wants the same definition to mean the same thing.

A `%%cql` cell is a definition, not a question, like a Python `def`. Running it checks the CQL and
declares the library; it reads no data. A Python cell runs a define when you want its result.

**Expected output:** `declared NotebookVisits|1.0.0 — include it, or run it from Python with cql(library="NotebookVisits")`

In [0]:
%%cql
library NotebookVisits version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Has Visit During Period":
  exists ([Encounter] E
    where E.period during "Measurement Period")

## 4. The measure — a second cell that `include`s the first
`include NotebookVisits` resolves against **this session**. The cell above is the library it
finds, and its text ships with the compile as a dependency, exactly as a file out of a Git Folder
would. Nothing was written to disk in between.

Like the first cell, it declares and runs nothing. It defines `Initial Population` and other checks,
which run when section 5 calls `exec()`.

**Expected output:** `declared NotebookMeasure|1.0.0 — include it, or run it from Python with cql(library="NotebookMeasure")`

In [0]:
%%cql
library NotebookMeasure version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers
include NotebookVisits version '1.0.0' called Visits

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Has Visit":
  Visits."Has Visit During Period"

define "Has Any Condition":
  exists ([Condition])

define "Initial Population":
  "Has Visit" and "Has Any Condition"

## 5. The draft is addressable by name
Both cells are libraries the session can list and bind, so the Python API reaches them the same way
it reaches published content. Python is where a cell's defines run, and where the measurement
period is bound to a different year.

Watch the population turn over: in 2025 no patient has both a visit and a condition. The patients seen
in 2025 have no condition on record. The measure that reports 1,223 of 4,000 for 2026 reports 0 of 4,000
for 2025 — the same content, honestly producing both numbers.

**One thing to know before you draft a second measure in the same notebook.** Every define is
materialized as a session temp view named after it, so two measures that both define `Initial Population`
overwrite each other's view and the older frame quietly starts returning the newer measure's rows.
Set `spark.conf.set("spark.cqldatabricks.objectPrefix", "auto")` before connecting to keep each
library's objects apart — worth doing the moment a session holds more than one measure, which is
exactly what authoring in a notebook invites.

**Expected output:**
```
declared: ['NotebookVisits|1.0.0', 'NotebookMeasure|1.0.0', 'FHIRHelpers|4.0.1', 'HypertensionBPControl|1.0.0', 'ColorectalScreening|1.0.0', 'MeasureCommon|1.0.0', 'DiabetesHighOrMissingA1c|1.0.0', 'AdultFluImmunization|1.0.0']
2026: initial_population=1223
2025: initial_population=0
```

In [0]:
from cql_databricks.compiler import VALUE_COLUMN

print("declared:", [str(r) for r in client.libraries()])

measure = client.cql(library="NotebookMeasure")

for year in (2026, 2025):
    pops = measure.with_measurement_period(f"{year}-01-01", f"{year}-12-31").exec()
    initial_pop = pops["Initial Population"].collect()
    count = sum(1 for r in initial_pop if r[VALUE_COLUMN])
    print(f"{year}: initial_population={count}")

## 6. Editing a definition
Re-running a cell **replaces** the library it declared. Make a simple edit to the visits library and
the measure that includes it answers differently on the next run — no re-import, no restart, no publish step.

That is also the one hazard worth naming out loud: **the session is the source of truth here, not the
page.** A cell you edited but never re-ran is not the definition the next compile sees. If a number
ever surprises you mid-draft, re-run the library cell before believing it.

The edit below widens the visits check to count a visit up to a year after the period ends. Run for 2025,
the measure counted no patients in section 5. With the edit it also counts the 2026 visits, and every
patient with a condition had one.

**Expected output:** `widened_2025: initial_population=1223`

In [0]:
# The same library, redeclared from Python rather than a cell: `register_cell` is what `%%cql` calls, so a
# generated or parameterized variant of a definition is a first-class way to declare one.
from pyspark.sql import functions as F

from cql_databricks.compiler import register_cell

draft = client.library_provider.source("NotebookVisits").cql
# Edit: count a visit up to a year after the period ends. Fail loud if the text to edit is not there.
ORIGINAL = 'E.period during "Measurement Period"'
WIDENED = 'E.period during Interval[start of "Measurement Period", end of "Measurement Period" + 1 year]'
if ORIGINAL not in draft:
    raise ValueError(f"NotebookVisits no longer contains {ORIGINAL!r}; update this edit")
register_cell(draft.replace(ORIGINAL, WIDENED))

edited_measure = client.cql(library="NotebookMeasure").with_measurement_period("2025-01-01", "2025-12-31")
initial_pop = edited_measure.exec("Initial Population").collect()
count = sum(1 for r in initial_pop if r[VALUE_COLUMN])
print("widened_2025: initial_population=", count, sep="")

## 7. Libraries in a folder
### Publishing the draft
A published library is the same text in a file. Writing the two cells out and mounting that directory
gives a folder like the examples' `measures/` folder below — and the measure resolves by name from it with
no change to a single line of CQL.

In a real workflow the directory is your measures repo as a **Databricks Git Folder**, and publishing
is a commit. The point here is only that nothing about the draft was notebook-shaped: the cell was
always the file.

**You publish the draft you have.** The previous section left `NotebookVisits` sitting with the
widened-window edit, and writing the session out now would publish exactly that — so the experiment
is put back first. Worth doing deliberately rather than discovering it in a file: the notebook holds
one current definition per name, not a history you can pick a version out of.

Cells still resolve ahead of the folder, so `clear()` is what hands the names back to the published
copies once both exist.

The reverse never happens. A measure resolved **from the folder** has its whole `include` closure
resolved there too, so a same-named cell cannot change what a published measure means — overriding is
something you opt into by writing the measure as a cell, never something that happens to a measure you
merely ran. A notebook that could quietly redefine published content would make every number out of it
impossible to cite.

**Expected output:**
```
published: ['NotebookMeasure-1.0.0.cql', 'NotebookVisits-1.0.0.cql']
from the folder: initial_population=1223
```

In [0]:
import pathlib
import tempfile

# Put the original NotebookVisits definition back; the widened window was an experiment.
register_cell(client.library_provider.source("NotebookVisits").cql.replace(WIDENED, ORIGINAL))

published = pathlib.Path(tempfile.mkdtemp(prefix="cql_databricks-draft-"))
for ref in client.library_provider.declared():
    (published / f"{ref.name}-{ref.version}.cql").write_text(
        client.library_provider.source(ref.name).cql)
print("published:", sorted(p.name for p in published.glob("*.cql")))

# Mount the folder, then drop the drafts so the published copies are what the names resolve to.
from_folder = connect(spark, library_path=str(published))
from_folder.library_provider.clear()

pops = from_folder.cql(library="NotebookMeasure").exec("Initial Population")
print(f"from the folder: initial_population={pops.filter(F.col(VALUE_COLUMN)).count()}")

### The examples' folder — the Git Folder setup
A real measure is not text you paste: it is a library that `include`s one or more others, published and
versioned in a repo.

The compile service resolves **none** of those includes — deliberately. It is a dumb endpoint: the
caller resolves the closure and ships every dependency, and the service only merges what it is
handed. It reads no filesystem, no registry and no network. So the resolver
lives on the client, next to the files.

`library_path` mounts one or more directories of `.cql` as the session's library source, and a
measure in them runs **by name**. In a real deployment that
directory is a **Databricks Git Folder** — your measures repo, synced by the platform — or a Unity
Catalog Volume. It is an ordinary directory either way, so there is no platform code involved:

```python
client = connect(spark, library_path="/Workspace/Users/me/my-measures/cql")
```

Roots are **ordered and earlier wins**, so a work-in-progress directory shadows a vendored content
repo: `library_path=["/Workspace/Users/me/wip", "/Volumes/acme/content/measures"]`.

Here we point it at the `measures/` folder published beside the example notebooks — the example measures, which the
deploy also published into the environment's knowledge. The path is derived from this notebook's own location, so the cell works unchanged
wherever the folder was copied to.

**Expected output:**
```
library_path: /Workspace/…/examples/measures
  AdultFluImmunization|1.0.0
  ColorectalScreening|1.0.0
  DiabetesHighOrMissingA1c|1.0.0
  FHIRHelpers|4.0.1
  HypertensionBPControl|1.0.0
  MeasureCommon|1.0.0
populations of HypertensionBPControl: ['Initial Population', 'Denominator', 'Denominator Exclusions', 'Numerator']
```
That last line is the whole feature. Binding by name resolved the measure, walked its `include`
closure to find the helper libraries, and shipped all of them to the compiler — and **no data was
touched**, because resolving and compiling a measure is not the same act as running it.

In [0]:
import pathlib

from cql_databricks.compiler import connect

_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
MEASURES = (pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.relative_to("/")
            / "measures")
if not MEASURES.is_dir():
    raise FileNotFoundError(f"no measures folder at {MEASURES} — it is published beside the example notebooks")

client = connect(spark, library_path=str(MEASURES))
print("library_path:", MEASURES)
for ref in client.libraries("*"):
    print(" ", ref)

measure = client.cql(library="HypertensionBPControl").with_measurement_period("2026-01-01", "2026-12-31")
print("populations of HypertensionBPControl:", measure.populations())

## 8. What did *not* happen
No endpoint. No token. No cluster. No `%scala` cell. No catalog or schema named anywhere in this
notebook. A single deployment matching the naming convention resolves itself:

| What | How it was resolved |
|---|---|
| Compiler Service endpoint | looked up in the workspace App registry by name |
| Authentication | your ambient identity, exchanged for an App token (RFC 8693) |
| The `cql_*` operator catalog | read out of `system.information_schema.routines` |
| FHIR data / value sets | the default environment: the catalog tagged `cql_databricks_environment_default`, whose `data` and `knowledge.valueset` tables are read by name — only when the CQL retrieves data |

Each is overridable (`connect(app_name=…, environment=…, data=…, terminology=…)`) — but you have to *want*
to.

The library source is the one thing you *do* name, and that is deliberate: which measures you mean
is a decision, not a convention.

### Where next
- **04 Inspection** — the define graph, the ELM and the Spark plan of a measure, read without running it.
- **05 Data and Knowledge Resolution** — every place data, value sets, libraries and results can come from.

Back to **00 Start Here** for the index.

In [0]:
# Every cell above ran, so the notebook is green. There are no value assertions here by design: the
# narrative states each expected result, and a compile or runtime failure fails its own cell loudly
# rather than being collected into a summary at the end.
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "03_authoring_libraries"}))